<p align="center">
<a href="https://duckietown.com"><img src="../../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

# 🚙 💻 Encoder-based Odometry

"Odometry" is the problem of "measuring the path", or evolution of the pose in time, of the robot. 

We can solve the odometry problem by using the measurements from wheel encoders and a kinematic model to estimate the evolution of the pose in time through an iterative procedure, such that:

<p style="text-align:center;"><img src="../../assets/images/odometry/odometry-1.png" width="500" alt="odometry-1"></p>   

$$ x_{k+1} = x_k + \Delta x_k $$
$$ y_{k+1} = y_k + \Delta y_k $$
$$ \theta_{k+1} = \theta_k + \Delta \theta_k $$

Where initial conditions ($x_0$, $y_0$, $\theta_0$) are assumed to be known. The increments can be calculated by:

1. **Determining the rotation of each wheel through the wheel encoder measurements**

$$\Delta \phi_k = N_k \cdot \alpha$$

where $N_k$ is the number of pulses, or "ticks", measured from the encoders in the $k$-th time interval, $\alpha = \frac{2 \pi}{N_{tot}}$ is the rotation per tick, and $N_{tot}$ is the total number of ticks per revolution ($N_{tot} = 135$ for the wheel encoders we will be using). This relation is evaluated for each wheel, yielding $\Delta \phi_{l,k}$ and $\Delta \phi_{r,k}$ for the left and right wheels respectively.

2. **Deriving the total distance travelled by each wheel**

<p style="text-align:center;"><img src="../../assets/images/odometry/odometry-d.png" width="250" alt="odometry-d"></p> 

Assuming the wheel radii are the same (equal to $R$) for both wheels, the distance travelled by each wheel is given by:

$$ d_{l/r, k} = R \cdot \Delta \phi_{l/r,k}$$

3. **Finding the rotation and distance travelled by the robot (robot frame)**

<p style="text-align:center;"><img src="../../assets/images/odometry/odometry-2.png" width="250" alt="odometry-2"></p>    

Under the assumption of no slipping of the robot wheels, we can derive the distance travelled by the origin of the robot frame (point $A$) and the rotation of the robot $\Delta \theta$:

$$ d_{A, k} = \frac{d_{r,k} + d_{l,k}}{2} $$
$$ \Delta \theta_{k} = \frac{d_{r,k} - d_{l,k}}{2L}$$

4. **Expressing the robot motion in the world reference frame $W$**

<p style="text-align:center;"><img src="../../assets/images/odometry/odometry-3.png" width="250" alt="odometry-3"></p>

Finally, we can express the estimated motion in the world reference frame and find:

$$ \Delta x_k = d_{A, k} \cos\theta_k $$
$$ \Delta y_k = d_{A, k} \sin\theta_k $$

# 🚙 💻 Let's get started!

You will write a function that produces an estimate of the pose of the Duckiebot, given measurements from the wheel encoders and an initial position:

In [ ]:
# Run and do not edit this magic cell. 
# It helps getting things to work throughout the Jupyter notebook - in particular importing changes in functions made in files other than this workspace.

%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

x0 = y0 = 0.0 # meters
theta0 = np.deg2rad(0) # radians

print(x0,y0, theta0)

## Determine the rotation of each wheel through the wheel encoder measurements

We have seen how to read wheel encoder data in the [wheel encoder tutorial](../02-Wheel-Encoders/wheel_encoders.ipynb). We can now use this data to measure the rotation of each wheel. 

### Wheel encoder parameters

Remember that there are 135 ticks per revolution on the wheel encoders we are using. 

In [ ]:
# Write the correct expressions 
import numpy as np 

N_tot = 135 # total number of ticks per revolution
alpha = 2 * np.pi / N_tot # wheel rotation per tick in radians

print(f"The angular resolution of our encoders is: {np.rad2deg(alpha)} degrees")

Assume that at the current update the left and right motor encoders have produced the following measurements:

In [ ]:
# Feel free to play with the numbers to get an idea of the expected outcome

ticks_left = 1
prev_tick_left = 0

ticks_right = 0
prev_tick_right = 0

By how many radians did each wheel rotate? 

In [ ]:
# How much would the wheels rotate with the above tick measurements? 

# Repetita iuvant: don't confuse degrees and radians when expressing angles
# Machines always use radians, humans make sense of degrees better. 
# Mixing these up is a very very common source of error!

delta_ticks_left = None # calculate delta ticks of left wheel 
delta_ticks_right = None # calculate delta ticks of right wheel 
rotation_wheel_left = None # calculate total rotation of left wheel 
rotation_wheel_right = None # calculate total rotation of right wheel 

print(f"The left wheel rotated: {np.rad2deg(rotation_wheel_left)} degrees")
print(f"The right wheel rotated: {np.rad2deg(rotation_wheel_right)} degrees")

## 🚙 💻 Evaluate distance travelled by each wheel

Now let's calculate the distance travelled by each wheel. It depends on the wheel radii. We need to determine them! We could use advanced odometry calibration procedures, but let's take it a step at the time. 

If you have a robot, take a ruler and measure your wheel radii (let's assume they are the same):

In [ ]:
# What is the radius of your wheels (assuming they are identical)? 

R = 0.0318 # you could double check your robot but this is probably correct

Note: the default value used in the Duckiematrix and on the robot is $R = 0.0318 \text{m}$.

In [ ]:
# What is the distance travelled by each wheel?

d_left = None # Calculate the distance traveled by the left wheel
d_right = None # Calculate the distance traveled by the right wheel

print(f"The left wheel travelled: {d_left} meters")
print(f"The right wheel travelled: {d_right} meters")

### 🚙 (Optional) Save your new value of `R`

If you have a Duckiebot, let's make sure it remembers its new wheel radius! 

The physical parameters of your robot (wheel radius, wheel baseline, encoder resolution) live in a calibration file **on the robot**: `/data/config/calibrations/robot/ROBOTNAME.yaml`. Every node that needs them, including the `encoder_pose_node` you will run later in this notebook and the `kinematics_node` of the robot, reads this file when it starts.

Power your Duckiebot on, make sure it is connected to the network and you can ping it, then open a terminal **on your computer** and type:

    ssh duckie@ROBOTNAME.local
    cd /data/config/calibrations/robot
    cp default.yaml ROBOTNAME.yaml     # only the first time: start from the default values
    nano ROBOTNAME.yaml

(use `sudo` in front of the last two commands if you get a permission error). Set the `wheel_radius` key to the value of `R` you measured (expressed in meters), the file looks like this:

    wheel_radius: 0.0318
    wheel_baseline: 0.1
    encoder_resolution: 135

then save and exit (in `nano`: `Ctrl+O`, `Enter`, `Ctrl+X`). The new values are read the next time the nodes start (e.g., the next time you run `dts code workbench`). 

**Note**: for a virtual robot the same file lives on your computer, under `~/.duckietown/virtual_robots/ROBOTNAME/root/data/config/calibrations/robot/`.

You can keep the terminal you just used open, so we can save the baseline measurement too. Let's keep going!


## 3. 🚙 💻 Find the rotation and distance travelled by the Duckiebot

If you have previously set your robot's gain so that the wheels do not slip, the travelled distance of point $A$ (origin of the robot frame) will be given by the average of the distances travelled by the wheels:

In [ ]:
# How much has the robot travelled? 

d_A = None # robot distance travelled in robot frame (meters)

print(f"The robot has travelled: {d_A} meters")

To calculate the rotation of the robot we need to measure the baseline too - or the distance between the center of the two wheels ($2L$): 

<p style="text-align:center;"><img src="../../assets/images/odometry/odometry-baseline.png" width="300" alt="odometry-baseline"></p>  

If you have a robot, take a ruler and measure it! 

In [ ]:
# What is the baseline length (2L) of your robot? 

baseline_wheel2wheel = 0.1 #  This is probably correct but feel free to double check 

Note: the default value, and that used in the Duckiematrix, is $baseline = 0.1m$.

We are now ready to calculate the rotation of the Duckiebot:

In [ ]:
# Of what angle has the robot rotated? 

Delta_Theta = None # [radians]

print(f"The robot has rotated: {np.rad2deg(Delta_Theta)} degrees")

### 🚙 (Optional) Save your new value of `baseline`

Let's make sure it remembers its new wheel baseline! 

Open (or go back to) the terminal connected to your Duckiebot, edit the same calibration file as above (`/data/config/calibrations/robot/ROBOTNAME.yaml` on the robot, `~/.duckietown/virtual_robots/ROBOTNAME/root/data/config/calibrations/robot/ROBOTNAME.yaml` for a virtual robot):

    nano /data/config/calibrations/robot/ROBOTNAME.yaml

and set the `wheel_baseline` key to the value of `baseline_wheel2wheel` you just measured (expressed in meters). Save and exit: the nodes will use the new baseline the next time they start.


# 💻 🚙 Write the odometry function

Now it is time to write the functions that will actually be running on the robot. 

You will write two functions:

1. A function that calculates the rotation of a wheel given a message from the wheel encoders and the previous number of ticks measured;

2. The actual odometry function, that will receive as inputs the kinematic model parameters, the pose estimate at the previous iteration, and the rotation of each wheel. The initial position is assumed to be ${\bf x}_0 = [0,0,0]^T$.

## Calculating the rotation of each wheel

Implement the function `delta_phi` inside the file [odometry.py](../../packages/odometry/odometry.py).

This function should output the wheel rotation (in radians) since last measurements, receiving as input the current and previous update wheel encoder readings. This function is called from a ROS 2 node [encoder_pose_node.py](../../packages/encoder_pose/encoder_pose/encoder_pose_node.py) which we have written, but you might want to take a look at it to make sure you understand what's going on. 


## Estimating the odometry

Similarly, implement the function `estimate_pose` inside the file [odometry.py](../../packages/odometry/odometry.py).
This function computes the `(x, y, theta)` estimate by aggregating computed wheel rotations and the (known) geometry of the robot.


### Test the `estimate_pose` function

Unit tests are useful to check if a piece of code does its intended job. Although the interaction of different functions might yield surprises even when each function produces the expected outcome, it is good practice to test them in isolation before prime time! These are called "unit tests", and:

> If it ain't tested, it's broken.
>
> --Roboticists, level 9

Let's see if the function you wrote above passes the following test!

In [ ]:
import sys
sys.path.append('../../tests')
sys.path.append('../../packages')
from unit_test import UnitTestOdometry

from odometry import estimate_pose


# UnitTestOdometry tests the `estimate_pose` function.
# The test is successful if you get a circle in the plot. 
# Anything different from a circle indicates that the odometry function has something wrong.

UnitTestOdometry(R, baseline_wheel2wheel, estimate_pose)


# 💻 🚙 Running Everything

Let's now see how the odometry estimation is working in practice. 



Follow the instructions in the [README](../../README.md) to build your code (`dts code build -R ROBOTNAME`) and run it on your robot (`dts code workbench -R ROBOTNAME`, or `dts code workbench -m -R ROBOTNAME` to test in the Duckiematrix). 

The terminal on your computer where you ran `dts code workbench` will be streaming some debugging data from your `encoder_pose_node`: the node publishes its _belief_ of the pose of the robot as a `nav_msgs/Odometry` message on the topic `/ROBOTNAME/encoder_pose_node/odom`, calculated _according to the `estimate_pose` and `delta_phi` functions written above_ (they are beliefs, not "real" states). 

Fill in the name of your robot and run the cells below. You will see three things:

- a plot of the trajectory of your robot according to your odometry (the orange arrow is the current pose estimate);
- what your robot sees;
- a virtual joystick (and a button that resets the pose estimate to the origin).

**Note**: it may take some time (>30s) for the images and the odometry to appear, depending on the specifications of your host machine.  


In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ActionButtons, ImageView, TopicPlot, TrajectoryPlot
from lx_kinematics_tools import Joystick

ROBOT = "ROBOT_NAME"   # <-- the name of your (real or virtual) robot
ros = RosSession(ROBOT)   # connects to the robot (its ROS 2 topics are now reachable from this notebook)


In [ ]:
# the odometry estimated by your encoder_pose_node, plotted live (x/y path + heading)
trajectory = TrajectoryPlot(ros, "encoder_pose_node/odom")


In [ ]:
# what your robot sees
camera = ImageView(ros, "camera_node/image/compressed")


In [ ]:
# drive the robot, and reset the pose estimate to (0, 0, 0) before a new loop
Joystick(ros)   # if the robot does not move: Joystick(ros, topic="car_cmd") (older robot images, see the README)
ActionButtons(ros, "encoder_pose_node", ["reset"])


Click on the buttons of the virtual joystick and start driving. You will see the marker move too according to the wheel encoder data that the robot is receiving. You can monitor these (and other) messages by following the procedure learned in the [wheel encoders tutorial](../02-Wheel-Encoders/wheel_encoders.ipynb). 

Drive a loop. Get back to the initial position and look at the resulting odometry. Is your robot's _belief_ accurate? Why, or why not? 

Do you want to modify your odometry functions before proceeding? Update the `estimate_pose` and `delta_phi` functions, re-build (`dts code build -R ROBOTNAME`) and re-launch `dts code workbench -R ROBOTNAME`. Press `reset` to start a new loop from the origin.

Tips: 

- `trajectory.clear()` empties the plot (the pose estimate of the node is not affected: use the `reset` button for that);

- every panel can be closed with its `.close()` method, e.g. `camera.close()`.

<div style="text-align:center;">
        <div style="text-align:center;">
            <img src="../../assets/images/odometry/sim-odom-complete.png" width="300" />
            <img src="../../assets/images/odometry/od-sim-tour.png" width="300" />
            <p>Odometry with different markers on different loops.</p>
        </div>
</div>


If you were testing in the Duckiematrix you should try it on your robot now! Don't worry if things are not as perfect on the real Duckiebot!

<div style="text-align:center;">
        <div style="text-align:center;">
            <img src="../../assets/images/odometry/odometry-real-2.png" width="300" alt="successful-test-odometry-real" />
            <img src="../../assets/images/odometry/less-good-odometry.png" width="300" />
            <p>DB21 Duckiebot good and less good odometries.</p>
        </div>
</div>